# ADVC W1e: TinyImageNet cold-read / loader probe (Kaggle, CPU work only)

After W1d (RESULTS_LOG R-004): the DataLoader read ~27 img/s the first time and ~700 img/s when the same images were read again. The likely cause is cold reads from the `/kaggle/input` mount, because `tiny_if` holds symlinks into it. That is a HYPOTHESIS. Here **every test uses images not read before** in this session, so caching cannot hide the effect.

**Before running:** Accelerator **GPU T4 x2** (so the disk/CPU match the real runs; the GPU is not used), Internet On, Add Input **albert1994/tinyimagenet**. Then **Save Version -> Save & Run All (Commit)**.

| Test | What |
|---|---|
| T1 | raw byte reads, 600 fresh files each, 1 / 8 / 32 / 64 threads (is it latency-bound?) |
| T2 | DataLoader, train transform, bs16, 600 fresh images, 2 / 4 / 8 workers |
| T3 | re-read of the T2 workers-4 images (cache effect) |
| T4 | pre-warm 600 fresh files with 32 threads, then DataLoader 4 workers |
| T5 | copy 2,000 fresh files to /kaggle/working with 32 threads |
| T6 | DataLoader, eval transform, bs32, 600 fresh val images, 4 workers |

Estimate (unmeasured): ~3-6 min of tests + ~2-4 min setup. Copy the last cell's output back to Claude.

In [ ]:
# 1. Setup (no bitsandbytes needed)
import os, subprocess, sys, time, glob, json
BRANCH = 'revision-w1'
W = '/kaggle/working/w1e'
os.makedirs(W, exist_ok=True)
if not os.path.isdir('/kaggle/working/ADVC'):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, 'https://github.com/Jmanav/ADVC.git', '/kaggle/working/ADVC'], check=True)
os.chdir('/kaggle/working/ADVC')
GIT = subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout.strip()
import torch
ACC = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU'
print('git:', GIT, '| torch', torch.__version__, '|', ACC, '| cpus', os.cpu_count())
SRC = '/kaggle/input/datasets/albert1994/tinyimagenet/tiny-imagenet-200'
if not os.path.isdir(SRC):
    cands = [p for p in glob.glob('/kaggle/input/**/tiny-imagenet-200', recursive=True)
             if os.path.isfile(os.path.join(p, 'val', 'val_annotations.txt'))]
    assert cands, 'TinyImageNet not found: attach albert1994/tinyimagenet'
    SRC = cands[0]
t0 = time.time()
if not os.path.isdir('/kaggle/working/tiny_if/val_if'):
    r = subprocess.run([sys.executable, 'scripts/prepare_tiny_imagenet.py', '--root', SRC, '--out', '/kaggle/working/tiny_if'], capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
PREP_S = round(time.time() - t0, 1)
assert os.path.isdir('/kaggle/working/tiny_if/train_if') and os.path.isdir('/kaggle/working/tiny_if/val_if')
print('prepare seconds:', PREP_S)

TIMES = {}
def sh(name, cmd):
    t0 = time.time()
    with open(f'{W}/{name}.log', 'w') as log:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            log.write(line); print(line, end='')
        p.wait()
    TIMES[name] = {'seconds': round(time.time() - t0, 1), 'exit': p.returncode}
    print('[' + name + ']', TIMES[name])

In [ ]:
%%writefile /kaggle/working/w1e/p7_io.py
# W1e: every test reads files not read before in this session (disjoint slices of one seeded permutation).
import json, os, shutil, subprocess, sys, time
from concurrent.futures import ThreadPoolExecutor
sys.path.insert(0, '.')
import torch
from torch.utils.data import DataLoader, Subset
from models.loader import load_config
from utils.datasets import build_eval_transform, build_train_transform, build_remapped_folder

cfg = load_config('configs/base.yaml')
N = 600
out = {'cpu_count': os.cpu_count(), 'chunk_images': N}
tr = build_remapped_folder(cfg['dataset']['train_dir'], build_train_transform(cfg), cfg)
va = build_remapped_folder(cfg['dataset']['val_dir'], build_eval_transform(cfg), cfg)
out['train_images'] = len(tr)
out['val_images'] = len(va)
p0 = tr.samples[0][0]
out['train_file_is_symlink'] = os.path.islink(p0)
out['train_file_target'] = os.path.realpath(p0)

def cmd(c):
    try:
        return subprocess.run(c, shell=True, capture_output=True, text=True, timeout=60).stdout.strip()[-800:]
    except Exception as e:
        return repr(e)

out['mount_input'] = cmd('mount | grep /kaggle/input | head -5')
out['df_working'] = cmd('df -h /kaggle/working | tail -1')
perm = torch.randperm(len(tr), generator=torch.Generator().manual_seed(0)).tolist()
cursor = [0]

def fresh(n):
    s = perm[cursor[0]:cursor[0] + n]
    cursor[0] += n
    return s

def read_bytes(path):
    with open(path, 'rb') as f:
        return len(f.read())

def dl_rate(ds, idx, bs, nw):
    dl = DataLoader(Subset(ds, idx), batch_size=bs, shuffle=False, num_workers=nw)
    it = iter(dl)
    next(it)                      # worker start-up + first batch excluded
    t = time.time(); n = 0
    for xb, _ in it:
        n += xb.shape[0]
    dt = time.time() - t
    return {'images': n, 'seconds': round(dt, 2), 'images_per_s': round(n / dt, 1)}

def show(k):
    print(k, out[k], flush=True)

for k in [1, 8, 32, 64]:
    paths = [tr.samples[i][0] for i in fresh(N)]
    t = time.time()
    with ThreadPoolExecutor(k) as ex:
        nbytes = sum(ex.map(read_bytes, paths))
    dt = time.time() - t
    key = 'T1_raw_read|threads' + str(k)
    out[key] = {'files': N, 'seconds': round(dt, 2), 'files_per_s': round(N / dt, 1), 'MB': round(nbytes / 1e6, 2)}
    show(key)

warm_idx = None
for nw in [2, 4, 8]:
    idx = fresh(N + 16)
    key = 'T2_train_cold|workers' + str(nw)
    out[key] = dl_rate(tr, idx, 16, nw)
    show(key)
    if nw == 4:
        warm_idx = idx
out['T3_train_warm_reread|workers4'] = dl_rate(tr, warm_idx, 16, 4)
show('T3_train_warm_reread|workers4')

idx = fresh(N + 16)
paths = [tr.samples[i][0] for i in idx]
t = time.time()
with ThreadPoolExecutor(32) as ex:
    list(ex.map(read_bytes, paths))
ws = time.time() - t
out['T4_prewarm32_then_loader4'] = {'prewarm_seconds': round(ws, 2), 'prewarm_files_per_s': round(len(paths) / ws, 1),
                                    'loader': dl_rate(tr, idx, 16, 4)}
show('T4_prewarm32_then_loader4')

idx = fresh(2000)
dst = '/kaggle/working/w1e/copytest'
os.makedirs(dst, exist_ok=True)

def cp(i):
    shutil.copyfile(tr.samples[i][0], os.path.join(dst, str(i) + '.JPEG'))   # follows the symlink

t = time.time()
with ThreadPoolExecutor(32) as ex:
    list(ex.map(cp, idx))
dt = time.time() - t
out['T5_copy32_to_working'] = {'files': len(idx), 'seconds': round(dt, 2), 'files_per_s': round(len(idx) / dt, 1)}
shutil.rmtree(dst)
show('T5_copy32_to_working')

vperm = torch.randperm(len(va), generator=torch.Generator().manual_seed(1)).tolist()
out['T6_val_cold|workers4'] = dl_rate(va, vperm[:N + 32], 32, 4)
show('T6_val_cold|workers4')
out['train_files_used'] = cursor[0]
print('P7JSON ' + json.dumps(out))
json.dump(out, open('/kaggle/working/w1e/p7_io.json', 'w'), indent=1)

In [ ]:
# 2. Run the probe (CPU)
sh('p7_io', [sys.executable, f'{W}/p7_io.py'])

In [ ]:
# 3. SUMMARY: copy everything this cell prints back to Claude (also /kaggle/working/w1e/summary.txt). Not truncated.
L = ['git: ' + GIT, 'torch ' + torch.__version__ + ' | ' + ACC + ' | cpus ' + str(os.cpu_count()), 'prepare seconds: ' + str(PREP_S)]
L.append('== wall times / exit codes ==')
L.append(json.dumps(TIMES))
p = f'{W}/p7_io.json'
L.append('== p7_io ==')
L.append(open(p).read() if os.path.exists(p) else 'no JSON; log tail:\n' + ''.join(open(f'{W}/p7_io.log').readlines()[-40:]))
text = '\n'.join(L)
open(f'{W}/summary.txt', 'w').write(text)
print(text)